# 05. Categorical dtype и ML-препроцессинг

Тема: как pandas хранит и обрабатывает категориальные данные под капотом — и почему это важно для ML.

| Инструмент | Зачем |
|---|---|
| `dtype='category'` / `pd.Categorical` | экономия памяти, ускорение groupby/sort |
| `.cat` accessor | управление словарём категорий |
| `from_codes` | восстановить Categorical из integer-кодов |
| `get_dummies` углублённо | OHE: drop_first, dtype, sparse, взаимодействие с Categorical |
| Ординальный Categorical | ordered=True — категория с порядком |
| Memory & speed | почему category быстрее object |

## 1. Что такое Categorical под капотом

Pandas хранит `object`-колонку как массив Python-строк — дорого.
`category` хранит **два массива**: `codes` (int8/16) + `categories` (словарь уникальных значений).

```
object:   ['Tech', 'Finance', 'Tech', 'Energy', 'Tech']  → 5 строк в памяти
category: codes=[0, 1, 0, 2, 0]  categories=['Energy','Finance','Tech']  → 1 словарь + 5 int
```

Выгода растёт с повторяемостью: 1M строк из 5 уникальных значений → ~8x экономия.

In [ ]:
import pandas as pd
import numpy as np
import sys

np.random.seed(42)
N = 100_000

# Симуляция большого датасета сигналов
sectors   = np.random.choice(['Tech', 'Finance', 'Energy', 'Healthcare', 'Consumer'], N)
ratings   = np.random.choice(['buy', 'hold', 'sell'], N)

df = pd.DataFrame({'sector': sectors, 'rating': ratings,
                   'return_d': np.round(np.random.normal(0, 0.02, N), 5)})

# Сравнение памяти: object vs category
mem_obj = df['sector'].memory_usage(deep=True)
mem_cat = df['sector'].astype('category').memory_usage(deep=True)

print(f'sector как object:   {mem_obj:,} байт')
print(f'sector как category: {mem_cat:,} байт')
print(f'Экономия:            {mem_obj / mem_cat:.1f}x')

# Конвертация
df['sector'] = df['sector'].astype('category')
df['rating'] = df['rating'].astype('category')
print(f'\ndtypes после конвертации:\n{df.dtypes}')

## 2. `.cat` accessor — управление словарём категорий

`.cat.categories` — словарь уникальных значений (Index)  
`.cat.codes` — целочисленные коды каждой строки  
`.cat.set_categories()` — задать/расширить словарь вручную  
`.cat.remove_unused_categories()` — убрать категории которых нет в данных  
`.cat.rename_categories()` — переименовать

In [ ]:
# Работаем с малым df для наглядности
tickers_raw = pd.Series(['AAPL', 'GOOG', 'AAPL', 'MSFT', 'GOOG', 'AAPL'], dtype='category')

print('Значения:  ', tickers_raw.values.tolist())
print('categories:', tickers_raw.cat.categories.tolist())  # словарь (алфавитный порядок)
print('codes:     ', tickers_raw.cat.codes.tolist())        # целые числа

# set_categories — добавить тикер которого нет в данных (важно для align'а с другим df)
extended = tickers_raw.cat.set_categories(['AAPL', 'GOOG', 'JPM', 'MSFT'])
print('\nРасширенный словарь:')
print(extended.cat.categories.tolist())  # JPM добавлен, хотя его нет в данных
print('value_counts (JPM = 0):')
print(extended.value_counts())

# remove_unused_categories — убрать лишнее после фильтрации
filtered = tickers_raw[tickers_raw.isin(['AAPL', 'GOOG'])]
print('\nПосле фильтрации — MSFT всё ещё в словаре:')
print(filtered.cat.categories.tolist())
print('После remove_unused_categories:')
print(filtered.cat.remove_unused_categories().cat.categories.tolist())

## 3. Ординальный Categorical — ordered=True

Обычный Categorical — **номинальный**: порядка нет, категории равноправны.  
`ordered=True` — **ординальный**: порядок задан явно, работают `<`, `>`, `min()`, `max()`.

В ML: ординальный categorical можно напрямую конвертировать в `.cat.codes` — коды уже несут правильный порядок. OHE не нужен.

In [ ]:
# Рейтинг аналитика — есть явный порядок
rating_order = pd.CategoricalDtype(
    categories=['strong_sell', 'sell', 'hold', 'buy', 'strong_buy'],
    ordered=True
)

ratings_s = pd.Series(
    ['buy', 'hold', 'strong_sell', 'sell', 'strong_buy', 'hold', 'buy'],
    dtype=rating_order
)
print('Ординальный рейтинг:')
print(ratings_s)
print('\ncodes (порядок кодов = порядок категорий):')
print(ratings_s.cat.codes.tolist())  # strong_sell=0, sell=1, hold=2, buy=3, strong_buy=4

# Сравнение работает!
print('\nТолько рейтинги >= buy:')
print(ratings_s[ratings_s >= 'buy'].tolist())

# Для ML: .cat.codes уже правильно закодированы — можно сразу в модель
df_ratings = pd.DataFrame({'rating': ratings_s, 'price_target': [200, 150, 80, 120, 250, 160, 210]})
df_ratings['rating_encoded'] = df_ratings['rating'].cat.codes  # 0..4 с нужным порядком
print('\nГотово для ML (без OHE):')
print(df_ratings)

## 4. `from_codes` — восстановить Categorical из кодов

Сценарий: данные хранятся в БД/файле как целые коды (экономия места),
словарь категорий — отдельно. `from_codes` собирает их обратно.

Также используется когда модель предсказала класс как int — нужно вернуть label.

In [ ]:
# Сценарий 1: данные из БД в виде кодов
sector_vocab = ['Consumer', 'Energy', 'Finance', 'Healthcare', 'Tech']  # словарь
sector_codes = np.array([4, 2, 4, 1, 0, 4, 3, 2, 1])                   # коды из БД

sectors_cat = pd.Categorical.from_codes(sector_codes, categories=sector_vocab)
print('Восстановленный Categorical из кодов:')
print(sectors_cat)

# Сценарий 2: модель вернула предсказанные классы как int → вернуть human-readable
signal_vocab = ['sell', 'hold', 'buy']  # 0=sell, 1=hold, 2=buy
model_predictions = np.array([2, 0, 1, 2, 2, 0, 1])  # output классификатора

predicted_labels = pd.Categorical.from_codes(model_predictions, categories=signal_vocab)
print('\nПредсказания модели → human-readable:')
print(predicted_labels)

# from_codes с ordered=True — для ординальных предсказаний
risk_vocab = ['low', 'medium', 'high', 'critical']
risk_codes = np.array([0, 2, 1, 3, 0, 1])
risk_cat = pd.Categorical.from_codes(risk_codes, categories=risk_vocab, ordered=True)
print('\nОрдинальный risk из кодов:')
print(risk_cat)
print('Максимальный риск:', risk_cat.max())

## 5. `get_dummies` углублённо — все параметры для ML

| Параметр | Что делает | Когда нужен |
|---|---|---|
| `prefix` | префикс колонок | всегда, чтобы не конфликтовали имена |
| `drop_first=True` | убрать первую категорию | линейные модели (мультиколлинеарность) |
| `dtype=int` / `float` | тип данных 0/1 | sklearn требует float или int |
| `dummy_na=True` | NaN → отдельная колонка | если пропуски несут смысл |
| `columns=[...]` | применить к нескольким колонкам df | удобнее чем вручную join |
| `sparse=True` | sparse matrix | 1000+ категорий (тикеры, коды) |

In [ ]:
# Датасет активов с несколькими категориальными признаками
assets = pd.DataFrame({
    'ticker':  ['AAPL', 'GOOG', 'JPM', 'XOM', 'MSFT', 'GS'],
    'sector':  ['Tech', 'Tech', 'Finance', 'Energy', 'Tech', 'Finance'],
    'rating':  ['buy', 'hold', 'buy', 'sell', None, 'hold'],  # есть NaN
    'return_d': [0.012, -0.005, 0.003, -0.021, 0.008, 0.015],
})
print('Исходный df:')
print(assets)

# 1. Базовый OHE — несколько колонок сразу через columns=[...]
dummies_basic = pd.get_dummies(assets, columns=['sector'], prefix='sec', dtype=int)
print('\nOHE для sector (columns= вместо ручного join):')
print(dummies_basic)

In [ ]:
# 2. drop_first=True для линейных моделей
# Удаляет первую категорию алфавитно — она кодируется как «все остальные = 0»
dummies_drop = pd.get_dummies(assets['sector'], prefix='sec', drop_first=True, dtype=int)
print('drop_first=True — убрана Energy (алфавитно первая):')
print(dummies_drop)
print('Energy кодируется как sec_Finance=0, sec_Tech=0')

# Если хочешь убрать КОНКРЕТНУЮ категорию (не первую алфавитно)
dummies_manual = pd.get_dummies(assets['sector'], prefix='sec', dtype=int).drop(columns='sec_Tech')
print('\nУбрана Tech вручную (reference category = Tech):')
print(dummies_manual)

In [ ]:
# 3. dummy_na=True — NaN как отдельный признак
# Вариант 1: NaN — информативный (аналитик не дал рейтинг = особый случай)
dummies_with_na = pd.get_dummies(assets['rating'], prefix='rtg', dummy_na=True, dtype=int)
print('dummy_na=True — NaN становится отдельной колонкой rtg_nan:')
print(dummies_with_na)

# Вариант 2: NaN — просто пропуск, заполнить до OHE
rating_filled = assets['rating'].fillna('hold')  # нейтральный дефолт
dummies_filled = pd.get_dummies(rating_filled, prefix='rtg', drop_first=True, dtype=int)
print('\nNaN заполнен hold, drop_first=True:')
print(dummies_filled)

In [ ]:
# 4. Categorical + get_dummies: словарь контролирует ПОРЯДОК и НАЛИЧИЕ колонок
# Важно: если в test set нет какой-то категории, OHE даст разные колонки → модель сломается

# Правильный паттерн: зафиксировать словарь через Categorical
known_sectors = ['Consumer', 'Energy', 'Finance', 'Healthcare', 'Tech']  # все возможные

train_sectors = pd.Categorical(
    ['Tech', 'Finance', 'Tech', 'Energy'],
    categories=known_sectors  # словарь зафиксирован!
)
test_sectors = pd.Categorical(
    ['Tech', 'Consumer', 'Finance'],  # Consumer не было в train — это ок, словарь тот же
    categories=known_sectors
)

train_ohe = pd.get_dummies(pd.Series(train_sectors), prefix='sec', dtype=int)
test_ohe  = pd.get_dummies(pd.Series(test_sectors),  prefix='sec', dtype=int)

print('Train OHE колонки:', train_ohe.columns.tolist())
print('Test  OHE колонки:', test_ohe.columns.tolist())
print('Колонки совпадают:', train_ohe.columns.tolist() == test_ohe.columns.tolist())
print('\nTest OHE (Consumer присутствует):')
print(test_ohe)

## 6. Categorical + groupby — скорость и observed

groupby по Categorical по умолчанию возвращает строку для **каждой категории из словаря**,
даже если данных нет. Параметр `observed=True` убирает пустые группы.

На больших данных groupby по category работает значительно быстрее чем по object.

In [ ]:
np.random.seed(0)

# Квартильные бины через qcut → Categorical с порядком
returns = np.random.normal(0.001, 0.02, 1000)
quartiles = pd.qcut(returns, q=4, labels=['Q1_worst', 'Q2', 'Q3', 'Q4_best'])

print('Тип quartiles:', type(quartiles))
print('ordered:', quartiles.cat.ordered)
print('categories:', quartiles.cat.categories.tolist())

# groupby по Categorical
results = (pd.Series(returns)
           .groupby(quartiles)
           .agg(['count', 'mean', 'std'])
           .rename(columns={'mean': 'avg_return', 'std': 'volatility'}))
results[['avg_return','volatility']] = results[['avg_return','volatility']].round(4)

print('\nСтатистика по квартилям доходности:')
print(results)

# observed=False — строки для всех категорий, даже пустых
small = pd.Series(returns[:10]).groupby(
    pd.qcut(returns[:10], q=4, labels=['Q1','Q2','Q3','Q4'])
)
print('\nobserved=False — видны пустые группы:')
print(small.agg('count'))

## 7. Итоговый паттерн: полный препроцессинг категорий для ML

In [ ]:
np.random.seed(7)
n = 12

raw = pd.DataFrame({
    'ticker':    np.random.choice(['AAPL','GOOG','MSFT','JPM','XOM'], n),
    'sector':    np.random.choice(['Tech','Finance','Energy'], n),
    'rating':    np.random.choice(['buy','hold','sell', None], n),
    'liquidity': np.random.choice(['high','medium','low'], n),  # ординальный
    'return_d':  np.round(np.random.normal(0, 0.02, n), 4),
})
print('RAW данные:')
print(raw)

# --- 1. Ординальные → .cat.codes (порядок важен) ---
liq_order = pd.CategoricalDtype(categories=['low','medium','high'], ordered=True)
raw['liquidity_enc'] = raw['liquidity'].astype(liq_order).cat.codes  # low=0,medium=1,high=2

# --- 2. Ординальный rating → .cat.codes ---
rating_order = pd.CategoricalDtype(categories=['sell','hold','buy'], ordered=True)
raw['rating'] = raw['rating'].fillna('hold')  # NaN → нейтральный
raw['rating_enc'] = raw['rating'].astype(rating_order).cat.codes

# --- 3. Номинальный sector → OHE с drop_first ---
sector_cat = pd.Categorical(raw['sector'], categories=['Energy','Finance','Tech'])
sector_ohe = pd.get_dummies(pd.Series(sector_cat), prefix='sec', drop_first=True, dtype=int)

# --- 4. Собрать финальную матрицу ---
feature_matrix = pd.concat([
    raw[['return_d', 'liquidity_enc', 'rating_enc']],
    sector_ohe
], axis=1)

print('\nFeature matrix (ML-ready):')
print(feature_matrix)
print(f'\nShape: {feature_matrix.shape}, dtypes:')
print(feature_matrix.dtypes.value_counts())